In [ ]:
import numpy as np

def gauss_seidel(A, b, x0, eps, n_max) -> tuple[np.ndarray, int, float]:

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.asarray(x0, dtype=float).copy()

    # Kiểm tra dữ liệu đầu vào
    if b.ndim != 1 or b.size == 0:
        raise ValueError("Đầu vào phải là vectơ không rỗng và các giá trị phải hữu hạn.")
    if not all(np.all(np.isfinite(v)) for v in (A, b, x)):
        raise ValueError("Đầu vào phải là vectơ không rỗng và các giá trị phải hữu hạn.")
    n = len(b)

    if A.shape != (n, n):
        raise ValueError("A phải là ma trận vuông có kích thước phù hợp với b.")

    if x.shape != (n,):
        raise ValueError("x0 phải có cùng số phần tử với b.")

    if not np.isscalar(eps) or not np.isfinite(eps) or eps <= 0:
        raise ValueError("eps phải lớn hơn 0.")

    if not isinstance(n_max, (int, np.integer)) or n_max <= 0:
        raise ValueError("n_max phải là số nguyên dương.")

    if np.any(np.diag(A) == 0):
        raise ValueError("Các phần tử trên đường chéo chính phải khác 0.")

    # Tính q theo từng hàng
    q_values = []

    for i in range(n):
        sigma = np.sum(np.abs(A[i, :])) - np.abs(A[i, i])
        q = sigma / np.abs(A[i, i])
        q_values.append(q)

        print(f"q{i + 1} = {q:.6f}")

    t = max(q_values)

    print(f"Giá trị q lớn nhất: {t:.6f}")

    # Kiểm tra điều kiện hội tụ
    htu = t < 1
    print("Điều kiện hội tụ:", htu)

    if not htu:
        raise ValueError("Không thỏa điều kiện hội tụ q < 1.")

    print()

    # Khởi tạo sai số
    saiso = float("inf")

    # Kiểm tra nghiệm ban đầu
    if np.max(np.abs(b - A @ x)) == 0:
        print("Nghiệm ban đầu đã thỏa mãn điều kiện dừng.")
        return x, 0, 0.0

    # Vòng lặp Gauss-Seidel
    for k in range(1, n_max + 1):

        x_old = np.copy(x)

        for i in range(n):

            # Dùng giá trị mới của x ở các vị trí trước i
            sum1 = np.dot(A[i, :i], x[:i])

            # Dùng giá trị cũ của x ở các vị trí sau i
            sum2 = np.dot(A[i, i + 1:], x_old[i + 1:])

            x[i] = (b[i] - sum1 - sum2) / A[i, i]

        # Tính sai số hậu nghiệm
        saiso = (t / (1 - t)) * np.max(np.abs(x - x_old))

        # Định dạng nghiệm đến 6 chữ số thập phân
        ketqua = np.array2string(
            x,
            formatter={'float_kind': lambda value: f'{value:.6f}'}
        )

        print(f"KQ lần lặp thứ {k}: {ketqua}")
        print(f"Sai số: {saiso:.6f}")

        # Kiểm tra điều kiện dừng
        if saiso < eps:
            print(f"Thỏa mãn sau {k} lần lặp.")
            return x, k, float(saiso)

    print(f"Đã thực hiện đủ {n_max} lần lặp.")
    return x, n_max, float(saiso)


# ==============================
# VÍ DỤ CHẠY THỬ
# ==============================

A = np.array([
    [9, 5, 2],
    [1, 7, 1],
    [-3, -1, -7]
], dtype=float)

b = np.array([1, 6, -26], dtype=float)

# Nghiệm ban đầu
x0 = np.array([0, 0, 0], dtype=float)

# Gọi hàm Gauss-Seidel
x, k, err = gauss_seidel(A, b, x0, eps=1e-5, n_max=3)

print("\n========== KẾT QUẢ ==========")
print("Nghiệm gần đúng:", np.round(x, 6))
print("Số lần lặp:", k)
print(f"Sai số: {err:.6f}")
